# LakeTwin-AI — Complete Colab Analysis Notebook

**Project:** *LakeTwin-AI: Big Data Analytics and Machine Learning for City-Scale Urban Lake Monitoring Using Satellite Imagery in Bengaluru*

This notebook starts from the existing seasonal lake-observation CSV exported from the Google Earth Engine (GEE) workflow. It performs data loading, quality checks, PySpark processing, exploratory K-Means clustering, poster-ready charts, and export of results.

> **Important scope note:** This notebook does not rerun Sentinel-2 image processing in GEE. It analyses the precomputed seasonal table. The GEE script and its lake-boundary asset are a separate upstream step. The study is a Bengaluru-area pilot covering 933 polygon features in the current table, not a verified census of every lake in Bengaluru. Candidate-water values are exploratory spectral proxies, not validated water area or lake-health scores.

## 0. Run instructions

1. Open this notebook in Google Colab.
2. Use **Runtime → Run all** (or run cells in order).
3. When prompted, upload `LakeTwin_AI_Combined_Seasonal_Observations_2020_2025.csv`.
4. The notebook saves analysis tables, cluster results, charts, and a ZIP bundle under `/content/LakeTwin_AI_Output/` and downloads a ZIP at the end if you choose.

The notebook deliberately uses `pdf`/`pyspark`-specific DataFrame names so the pandas `df` and Spark DataFrame cannot accidentally overwrite each other.

In [ ]:
# 1. Install and import dependencies
!pip -q install pyspark scikit-learn

import os, glob, shutil, warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
OUTPUT_DIR = Path("/content/LakeTwin_AI_Output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR)

In [ ]:
# 2. Find the combined CSV; upload it if it is not already in the Colab session
EXPECTED_NAME = "LakeTwin_AI_Combined_Seasonal_Observations_2020_2025.csv"
candidates = glob.glob(f"/content/**/{EXPECTED_NAME}", recursive=True)

if candidates:
    CSV_PATH = candidates[0]
    print("Found existing file:", CSV_PATH)
else:
    from google.colab import files
    print(f"Please upload {EXPECTED_NAME}")
    uploaded = files.upload()
    uploaded_csvs = [name for name in uploaded if name.lower().endswith(".csv")]
    if not uploaded_csvs:
        raise FileNotFoundError("No CSV uploaded. Upload the combined seasonal observations CSV.")
    # Prefer the expected filename; otherwise use the first uploaded CSV and validate its columns below.
    CSV_PATH = next((name for name in uploaded_csvs if os.path.basename(name) == EXPECTED_NAME), uploaded_csvs[0])
    print("Using uploaded file:", CSV_PATH)

# Pandas dataframe used for analysis throughout this notebook
pdf = pd.read_csv(CSV_PATH)
print("Loaded shape:", pdf.shape)
print("Columns:", pdf.columns.tolist())
pdf.head()

In [ ]:
# 3. Validate required columns and normalize types
REQUIRED_COLUMNS = [
    "source_idx", "lake_name", "year", "season", "image_count",
    "period_start", "period_end", "NDWI", "MNDWI", "NDVI", "candidate_water"
]
missing_columns = [c for c in REQUIRED_COLUMNS if c not in pdf.columns]
if missing_columns:
    raise ValueError(
        "The uploaded CSV is missing required columns: " + str(missing_columns) +
        "\nPlease upload the combined seasonal observations CSV, not a single-scene or one-season export."
    )

for col in ["source_idx", "year", "image_count"]:
    pdf[col] = pd.to_numeric(pdf[col], errors="coerce")
for col in ["NDWI", "MNDWI", "NDVI", "candidate_water"]:
    pdf[col] = pd.to_numeric(pdf[col], errors="coerce")
pdf["lake_name"] = pdf["lake_name"].astype("string").str.strip()
pdf["season"] = pdf["season"].astype("string").str.strip()

print("Rows:", len(pdf))
print("Year range:", pdf["year"].min(), "to", pdf["year"].max())
print("Unique source indices:", pdf["source_idx"].nunique())
print("Unique non-empty lake names:", pdf["lake_name"].replace("", pd.NA).nunique())
print("Exact duplicate rows:", int(pdf.duplicated().sum()))
pdf.head()

In [ ]:
# 4. Data-quality summary
spectral_cols = ["NDWI", "MNDWI", "NDVI", "candidate_water"]
quality = pd.DataFrame({
    "metric": [
        "total_records", "complete_spectral_records", "records_missing_any_spectral_value",
        "exact_duplicate_rows", "unique_source_indices", "unique_lake_names"
    ],
    "value": [
        len(pdf), int(pdf[spectral_cols].notna().all(axis=1).sum()),
        int(pdf[spectral_cols].isna().any(axis=1).sum()), int(pdf.duplicated().sum()),
        int(pdf["source_idx"].nunique()), int(pdf["lake_name"].replace("", pd.NA).nunique())
    ]
})
print(quality.to_string(index=False))
quality.to_csv(OUTPUT_DIR / "data_quality_summary.csv", index=False)

print("\nMissing values by field:")
print(pdf.isna().sum().to_string())
print("\nRecords by year and season:")
coverage = (pdf.groupby(["year", "season"], dropna=False)
            .agg(records=("source_idx", "size"), unique_polygons=("source_idx", "nunique"),
                 mean_image_count=("image_count", "mean"),
                 complete_spectral_records=("candidate_water", "count"))
            .reset_index().sort_values(["year", "season"]))
display(coverage)
coverage.to_csv(OUTPUT_DIR / "year_season_coverage.csv", index=False)

## 5. Data interpretation checks

- `candidate_water` is expected to be a proportion-like proxy between 0 and 1 in this dataset. The check below reports out-of-range values rather than silently clipping them.
- Missing spectral values are excluded from the ML step, not imputed.
- Repeated names or multiple polygon indices are flagged for review but are **not merged automatically** because geometry is needed to determine whether they are aliases, separate polygons, or duplicates.

In [ ]:
# 5. Plausibility checks and polygon-name audit
cw = pdf["candidate_water"].dropna()
print("candidate_water min/max:", (cw.min(), cw.max()) if len(cw) else "no valid values")
print("candidate_water outside [0, 1]:", int(((cw < 0) | (cw > 1)).sum()))

names_per_index = pdf.groupby("source_idx", dropna=True)["lake_name"].nunique(dropna=True)
indices_per_name = pdf.groupby("lake_name", dropna=True)["source_idx"].nunique(dropna=True)
print("Source indices associated with multiple names:", int((names_per_index > 1).sum()))
print("Exact lake names associated with multiple source indices:", int((indices_per_name > 1).sum()))
repeated_names = indices_per_name[indices_per_name > 1].index
name_audit = (pdf[pdf["lake_name"].isin(repeated_names)]
              [["lake_name", "source_idx"]].drop_duplicates()
              .sort_values(["lake_name", "source_idx"]))
display(name_audit.head(100))
name_audit.to_csv(OUTPUT_DIR / "repeated_lake_name_audit.csv", index=False)

## 6. PySpark processing

This section independently loads the same CSV with Spark, removes rows missing the spectral fields used in this analysis, and produces a year-season summary. The summary is descriptive and unweighted across polygon records; it is not an area-weighted citywide lake estimate. A local Colab Spark session demonstrates use of Spark APIs, but it is not a benchmark of cluster-scale speed or performance gains.

In [ ]:
# 6. PySpark: clean and aggregate the seasonal records
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("LakeTwinAI_SeasonalAnalysis")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

spark_df = (spark.read.option("header", True).option("inferSchema", True).csv(CSV_PATH))
print("Spark input records:", spark_df.count())

spark_clean = spark_df.dropna(subset=["NDWI", "MNDWI", "NDVI", "candidate_water"])
print("Spark complete spectral records:", spark_clean.count())
print("Spark excluded incomplete records:", spark_df.count() - spark_clean.count())

spark_summary = (spark_clean.groupBy("year", "season")
    .agg(
        F.count("*").alias("valid_records"),
        F.countDistinct("source_idx").alias("unique_polygons"),
        F.round(F.avg("candidate_water"), 4).alias("mean_candidate_water_proxy"),
        F.round(F.expr("percentile_approx(candidate_water, 0.5)"), 4).alias("median_candidate_water_proxy"),
        F.round(F.avg("NDWI"), 4).alias("mean_NDWI"),
        F.round(F.avg("MNDWI"), 4).alias("mean_MNDWI"),
        F.round(F.avg("NDVI"), 4).alias("mean_NDVI"),
        F.round(F.avg("image_count"), 2).alias("mean_image_count")
    ).orderBy("year", "season"))

display(spark_summary)
spark_summary.toPandas().to_csv(OUTPUT_DIR / "pyspark_year_season_summary.csv", index=False)

## 7. Exploratory machine learning: K-Means clustering

K-Means groups rows by similarity in standardized NDWI, MNDWI and NDVI. The algorithm is unsupervised: the resulting cluster numbers do **not** inherently mean good/bad lake health, clean/polluted water, or confirmed water/non-water. The silhouette score is a cluster-separation diagnostic, not classification accuracy or environmental validation.

In [ ]:
# 7. Fit exploratory K-Means model
FEATURES = ["NDWI", "MNDWI", "NDVI"]
ml_df = pdf.dropna(subset=FEATURES).copy()
X = ml_df[FEATURES].astype(float)

if len(ml_df) < 4:
    raise ValueError("Not enough complete rows to run a meaningful exploratory clustering step.")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

N_CLUSTERS = 3
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
ml_df["cluster"] = kmeans.fit_predict(X_scaled)

cluster_sizes = ml_df["cluster"].value_counts().sort_index().rename("observations").reset_index()
cluster_sizes.columns = ["cluster", "observations"]
cluster_profiles = (ml_df.groupby("cluster")[FEATURES + ["candidate_water"]]
                    .mean(numeric_only=True).round(4).reset_index())

print("Rows used for ML:", len(ml_df))
print("Rows excluded for missing one or more ML features:", len(pdf) - len(ml_df))
print("\nCluster sizes:")
display(cluster_sizes)
print("\nMean original-scale feature values by cluster:")
display(cluster_profiles)

# Silhouette can be expensive on very large tables; use a reproducible sample if needed.
if ml_df["cluster"].nunique() > 1:
    sample_n = min(5000, len(ml_df))
    if len(ml_df) > sample_n:
        rng = np.random.default_rng(RANDOM_STATE)
        sample_idx = rng.choice(len(ml_df), size=sample_n, replace=False)
        sil = silhouette_score(X_scaled[sample_idx], ml_df["cluster"].to_numpy()[sample_idx])
        print(f"Silhouette score (reproducible sample n={sample_n}):", round(float(sil), 4))
    else:
        sil = silhouette_score(X_scaled, ml_df["cluster"])
        print("Silhouette score:", round(float(sil), 4))
else:
    sil = np.nan
    print("Silhouette score unavailable: model produced fewer than two non-empty clusters.")

cluster_sizes.to_csv(OUTPUT_DIR / "kmeans_cluster_sizes.csv", index=False)
cluster_profiles.to_csv(OUTPUT_DIR / "kmeans_cluster_profiles.csv", index=False)
ml_df.to_csv(OUTPUT_DIR / "LakeTwin_AI_KMeans_Results.csv", index=False)
pd.DataFrame({"metric": ["silhouette_score", "rows_used", "n_clusters"],
              "value": [sil, len(ml_df), ml_df["cluster"].nunique()]
             }).to_csv(OUTPUT_DIR / "kmeans_diagnostics.csv", index=False)
print("Saved K-Means row-level results and summaries.")

## 8. Poster-ready charts

The charts are generated directly from the CSV and are saved as PNG files. They use descriptive labels so the candidate-water proxy is not mistaken for validated lake water area.

In [ ]:
# 8A. Mean candidate-water proxy by season and year
season_order = ["PreMonsoon", "Monsoon", "PostMonsoon"]
plot_df = pdf.dropna(subset=["year", "season", "candidate_water"]).copy()
plot_df["season"] = plot_df["season"].astype(str)
season_summary = (plot_df.groupby(["year", "season"])["candidate_water"]
                  .agg(mean="mean", median="median", count="count").reset_index())

fig, ax = plt.subplots(figsize=(10, 5.5))
for season in season_order:
    sub = season_summary[season_summary["season"] == season].sort_values("year")
    if not sub.empty:
        ax.plot(sub["year"], sub["mean"], marker="o", linewidth=2, label=season)
ax.set_title("Seasonal candidate-water spectral proxy (2020–2025)")
ax.set_xlabel("Year")
ax.set_ylabel("Mean candidate-water proxy (0–1; exploratory)")
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)
ax.legend(title="Season")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_seasonal_candidate_water_trend.png", dpi=220, bbox_inches="tight")
plt.show()

season_summary.to_csv(OUTPUT_DIR / "seasonal_candidate_water_summary.csv", index=False)

In [ ]:
# 8B. Seasonal distribution: median proxy across years
season_distribution = (plot_df.groupby("season")["candidate_water"]
                       .apply(list).to_dict())
ordered = [s for s in season_order if s in season_distribution]
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.boxplot([season_distribution[s] for s in ordered], tick_labels=ordered, showfliers=False)
ax.set_title("Distribution of candidate-water proxy by season")
ax.set_xlabel("Season")
ax.set_ylabel("Candidate-water proxy (exploratory; not validated water area)")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_candidate_water_season_distribution.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
# 8C. Data completeness figure
complete_n = int(pdf[FEATURES + ["candidate_water"]].notna().all(axis=1).sum())
missing_n = int(len(pdf) - complete_n)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(["Complete spectral rows", "Rows with missing spectral data"], [complete_n, missing_n])
ax.set_title("Input data completeness")
ax.set_ylabel("Number of records")
ax.tick_params(axis="x", rotation=10)
for i, v in enumerate([complete_n, missing_n]):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_data_completeness.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
# 8D. K-Means cluster profiles (means of original-scale indices)
fig, ax = plt.subplots(figsize=(8, 5))
cluster_profiles.set_index("cluster")[FEATURES].plot(kind="bar", ax=ax)
ax.set_title("Exploratory K-Means cluster profiles")
ax.set_xlabel("Cluster ID (arbitrary; not a lake-health class)")
ax.set_ylabel("Mean spectral index")
ax.grid(axis="y", alpha=0.3)
ax.legend(title="Feature")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_kmeans_cluster_profiles.png", dpi=220, bbox_inches="tight")
plt.show()

## 9. Export a compact poster-results bundle

This packages the tables, model outputs and figures. It does not include the original satellite imagery or the GEE boundary asset.

In [ ]:
# 9. Create ZIP bundle of analysis outputs
zip_base = "/content/LakeTwin_AI_Poster_Analysis_Bundle"
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print("Bundle created:", zip_path)
print("Files included:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name, f"({p.stat().st_size:,} bytes)")

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Automatic download was not available. ZIP remains at:", zip_path, "Details:", e)

## 10. Safe claims for the poster

Use only claims supported by the generated outputs:

- The workflow analysed seasonal Sentinel-2-derived spectral summaries for the years represented in the input CSV.
- PySpark was used to clean and aggregate the seasonal table.
- K-Means was used for exploratory grouping of NDWI, MNDWI and NDVI observations, if the ML cell completes successfully.
- Candidate-water proxy patterns are descriptive and require validation against independent reference data.

Do **not** claim classification accuracy, measured speedup, validated lake-health classes, a verified census of all Bengaluru lakes, or precise water-area change unless you add independent evidence for those claims.

### Scope boundary

The upstream GEE workflow must already have created the combined seasonal CSV. If you want to regenerate satellite observations from scratch, the exact GEE boundary asset/table and GEE script are required; this analysis notebook cannot infer or recreate those source geometries from the CSV alone.